# **Desafío: trabajando en otros contextos**

Pongamos nuevamente en práctica todo lo que aprendimos durante la clase. He puesto los 2 nuevos conjuntos de datos disponibles para descargar a continuación:

- Proyecto Desafío 1: Ventas Online - dados_vendas_clientes.json;
- Proyecto Desafío 2: Administración de Condominios - dados_locacao_imoveis.json.
**Recuerda**: Estos dos proyectos de tratamiento se construyeron durante el curso. Por lo tanto, considere desarrollos previos para realizar este paso final.

**Etapa 4**

- **Proyecto Desafío 1: Ventas Online**

En los pasos anteriores ya hemos trabajado con varios tipos de datos, ahora podemos trabajar con datos de tiempo.

En la columna ```Fecha de venta``` tenemos fechas en el formato 'día/mes/año' (```dd/mm/AAAA```). Transforme estos datos al tipo datetime y busque una forma de visualización de subconjunto que pueda contribuir al objetivo del contexto en el que se insertan los datos.

Si no recuerdas el problema del Proyecto Desafío 1, te dejo el texto de la situación a continuación para que sea más fácil encontrar la información:

```
El objetivo de este proyecto es analizar los resultados de un evento con los clientes de una empresa de venta online. Se recopiló un conjunto de datos que contiene los clientes que gastaron más en productos dentro de los 5 días posteriores a la venta, que es la duración del evento. Este análisis identificará al cliente con la mayor compra esa semana, quien recibirá un premio de la tienda, y posteriormente, puede ayudar a la empresa a crear nuevas estrategias para atraer más clientes.
```
- **Proyecto Desafío 2: Administración de Condominios**

Al igual que en el Proyecto Desafío 1, trabajamos con todas las columnas excepto las que involucran fechas.

En las columnas ```datas_de_pagamento``` y ```datas_combinadas_pagamento``` tenemos fechas en el formato 'día/mes/año' (```dd/mm/AAAA```). Transforme estos datos al tipo datetime y busque una forma de visualización de subconjunto que pueda contribuir al objetivo del contexto en el que se insertan los datos.

Si no recuerdas el problema del Proyecto Desafío 2, te dejo el texto de la situación a continuación para que sea más fácil encontrar la información:
```
Administrar condominios es una tarea que requiere mucha atención y organización. Entre las diversas responsabilidades de gestión se encuentra el cobro del alquiler a los inquilinos. Para garantizar la buena salud financiera de la empresa, es fundamental que estos pagos se realicen de forma regular y puntual. Sin embargo, sabemos que esto no siempre sucede. Teniendo esto en cuenta, propongo un desafío de procesamiento de datos con el objetivo de analizar el retraso en el pago de la renta en el condominio ficticio de algunos residentes.
```

---

In [17]:
import pandas as pd
import json
import numpy as np

In [18]:
ventas_path = "dados_vendas_clientes.json"
administracion_path = "dados_locacao_imoveis.json"

ventas_json = json.load(open(ventas_path))
administracion_json = json.load(open(administracion_path))

In [19]:
ventas = pd.json_normalize(ventas_json["dados_vendas"])
ventas.head()

,Data de venda,Cliente,Valor da compra
0,06/06/2022,"[@ANA _LUCIA 321, DieGO ARMANDIU 210, DieGO AR...","[R$ 836,5, R$ 573,33, R$ 392,8, R$ 512,34]"
1,07/06/2022,"[Isabely JOanes 738, Isabely JOanes 738, Isabe...","[R$ 825,31, R$ 168,07, R$ 339,18, R$ 314,69]"
2,08/06/2022,"[Isabely JOanes 738, JOãO Gabriel 671, Julya m...","[R$ 682,05, R$ 386,34, R$ 622,65, R$ 630,79]"
3,09/06/2022,"[Julya meireles 914, MaRIA Julia 444, MaRIA Ju...","[R$ 390,3, R$ 759,16, R$ 334,47, R$ 678,78]"
4,10/06/2022,"[MaRIA Julia 444, PEDRO PASCO 812, Paulo castr...","[R$ 314,24, R$ 311,15, R$ 899,16, R$ 885,24]"


In [28]:
colummnas_ventas = ventas.columns.to_list()
ventas_explode = ventas.explode(colummnas_ventas[1:]).reset_index(drop=True)
print(ventas_explode.info())
ventas_explode["Cliente"] = ventas_explode["Cliente"].str.lower().replace("[^a-z]", "", regex=True)
ventas_explode["Valor da compra"] = ventas_explode["Valor da compra"].apply(lambda x: x.replace("R$","").replace(",",".").strip())
ventas_explode["Valor da compra"] = ventas_explode["Valor da compra"].astype(np.float64)
ventas_explode["Data de venda"] = pd.to_datetime(ventas_explode["Data de venda"], format='%d/%m/%Y')
print(ventas_explode.info())
print(ventas_explode.head())
total_compras = ventas_explode.groupby(['Cliente'])['Valor da compra'].sum().sort_values(ascending=False)
total_compras

<class 'pandas.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 3 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   Data de venda    20 non-null     str  
 1   Cliente          20 non-null     str  
 2   Valor da compra  20 non-null     str  
dtypes: str(3)
memory usage: 612.0 bytes
None
<class 'pandas.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 3 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   Data de venda    20 non-null     datetime64[us]
 1   Cliente          20 non-null     str           
 2   Valor da compra  20 non-null     float64       
dtypes: datetime64[us](1), float64(1), str(1)
memory usage: 612.0 bytes
None
  Data de venda        Cliente  Valor da compra
0    2022-06-06       analucia           836.50
1    2022-06-06  diegoarmandiu           573.33
2    2022-06-06  diegoarmandiu           392.80
3    2022-06-06  d

Cliente
isabelyjoanes    2329.30
mariajulia       2086.65
julyameireles    1643.74
diegoarmandiu    1478.47
paulocastro       899.16
thiagofritzz      885.24
analucia          836.50
joogabriel        386.34
pedropasco        311.15
Name: Valor da compra, dtype: float64

In [21]:
administracion = pd.json_normalize(administracion_json["dados_locacao"])
administracion.head()

,apartamento,datas_combinadas_pagamento,datas_de_pagamento,valor_aluguel
0,A101 (blocoAP),"[01/06/2022, 01/07/2022]","[05/06/2022, 03/07/2022]","[$ 1000,0 reais, $ 2500,0 reais]"
1,A102 (blocoAP),"[02/06/2022, 02/07/2022]","[02/06/2022, 06/07/2022]","[$ 1100,0 reais, $ 2600,0 reais]"
2,B201 (blocoAP),"[03/06/2022, 03/07/2022]","[07/06/2022, 03/07/2022]","[$ 1200,0 reais, $ 2700,0 reais]"
3,B202 (blocoAP),"[04/06/2022, 04/07/2022]","[07/06/2022, 05/07/2022]","[$ 1300,0 reais, $ 2800,0 reais]"
4,C301 (blocoAP),"[05/06/2022, 05/07/2022]","[10/06/2022, 09/07/2022]","[$ 1400,0 reais, $ 2900,0 reais]"


In [29]:
colummnas_administracion = administracion.columns.tolist()
administracion_explode = administracion.explode(colummnas_administracion[1:])
print(administracion_explode.info())
administracion_explode["apartamento"] = administracion_explode["apartamento"].apply(lambda x: x.replace("(blocoAP)","").strip())
administracion_explode["valor_aluguel"] = administracion_explode["valor_aluguel"].apply(lambda x: x.replace("$","").replace(",",".").replace("reais","").strip())
administracion_explode["valor_aluguel"] = administracion_explode["valor_aluguel"].astype(np.float64)
administracion_explode['datas_combinadas_pagamento'] = pd.to_datetime(administracion_explode['datas_combinadas_pagamento'], format='%d/%m/%Y')
administracion_explode['datas_de_pagamento'] = pd.to_datetime(administracion_explode['datas_de_pagamento'], format='%d/%m/%Y')
print(administracion_explode.info())
print(administracion_explode.head())
administracion_explode['atraso'] = (administracion_explode['datas_de_pagamento'] - administracion_explode['datas_combinadas_pagamento']).dt.days
avg_atraso = administracion_explode.groupby(['apartamento'])['atraso'].mean()
avg_atraso


<class 'pandas.DataFrame'>
Index: 30 entries, 0 to 14
Data columns (total 4 columns):
 #   Column                      Non-Null Count  Dtype
---  ------                      --------------  -----
 0   apartamento                 30 non-null     str  
 1   datas_combinadas_pagamento  30 non-null     str  
 2   datas_de_pagamento          30 non-null     str  
 3   valor_aluguel               30 non-null     str  
dtypes: str(4)
memory usage: 1.2 KB
None
<class 'pandas.DataFrame'>
Index: 30 entries, 0 to 14
Data columns (total 4 columns):
 #   Column                      Non-Null Count  Dtype         
---  ------                      --------------  -----         
 0   apartamento                 30 non-null     str           
 1   datas_combinadas_pagamento  30 non-null     datetime64[us]
 2   datas_de_pagamento          30 non-null     datetime64[us]
 3   valor_aluguel               30 non-null     float64       
dtypes: datetime64[us](2), float64(1), str(1)
memory usage: 1.2 KB
None
 

apartamento
A101    3.0
A102    2.0
B201    2.0
B202    2.0
C301    4.5
C302    4.0
D401    1.0
D402    4.0
E501    0.5
E502    4.0
F601    4.0
F602    1.5
G701    6.5
G702    2.0
H801    2.0
Name: atraso, dtype: float64